In [22]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

# ==========================================
# 1. GLOBAL CONFIGURATION & STYLING
# ==========================================
hep.style.use(hep.style.CMS)

COFFEA_DIR = "DataVsMC/" # <-- UPDATE THIS PATH
LUMI = 109.950           # target lumi in fb^-1 
TTBB_KFACTOR = 1.318

# --- DATA TOGGLE ---
# Set to False for MC-only: Plots Main, S/B, and S/sqrt(B)
# Set to True to unblind: Plots Main, S/B, and Data/MC ratio
PLOT_DATA = False  

# --- DICTIONARIES ---
TARGET_XSECS = {
    'tttolnu2q': 405.46, 'ttto4q': 419.31, 'ttto2l2nu': 97.90,
    'ttbbtolnu2q': 17.37, 'ttbbto4q': 19.15, 'ttbbto2l2nu': 3.75,
    'tth-hto2b': 0.331, 'tth-htonon2b': 0.238, 'ttz-ztoqq-1jets': 0.786,
    'ttwjets': 0.468, 'ttll_bin-mll-4to50': 0.039, 'ttll_bin-mll-50': 0.086, 'ttnunu': 0.164,
    'tttt': 0.0097, 'thw': 0.017, 'thq': 0.084, 'tzq_ll': 0.080,
    'st_tw_top-4q': 19.95, 'st_tw_antitop-4q': 19.95, 'st_tw_top-lnu2q': 19.29, 'st_tw_antitop-lnu2q': 19.29,
    'st_tw_top-2l2nu': 4.66, 'st_tw_antitop-2l2nu': 4.66, 'st_top_s_lep': 2.278, 'st_antitop_s_lep': 1.43,
    'st_top_t_2q': 97.74, 'st_antitop_t_2q': 58.78, 'st_top_t_lnu': 47.26, 'st_antitop_t_lnu': 28.42,
    'wtolnu-4jets_bin-1j': 9141.0, 'wtolnu-4jets_bin-2j': 2931.0, 'wtolnu-4jets_bin-3j': 864.6, 'wtolnu-4jets_bin-4j': 417.8,
    'dyto2e_mll-10to50': 5743.0, 'dyto2e_mll-50': 1827.0, 'dyto2mu_mll-10to50': 5803.0, 'dyto2mu_mll-50': 1815.0,
    'dyto2tau_mll-10to50': 5806.0, 'dyto2tau_mll-50': 1816.0,
    'ww': 125.8, 'wz': 54.7, 'zz': 16.7,
    'qcd-ht_40_70': 312300000.0, 'qcd-ht_70_100': 58470000.0, 'qcd-ht_100_200': 25310000.0,
    'qcd-ht_200_400': 1960000.0, 'qcd-ht_400_600': 97400.0, 'qcd-ht_600_800': 13560.0,
    'qcd-ht_800_1000': 3010.0, 'qcd-ht_1000_1200': 890.3, 'qcd-ht_1200_1500': 384.8,
    'qcd-ht_1500_2000': 127.3, 'qcd-ht_2000': 26.26
}

TARGET_GENWEIGHTS = {
    'tttolnu2q': 480547550.0, 'ttto2l2nu': 466318140.0, 'ttto4q': 468705400.0,
    'ttbbtolnu2q': 21007254.0, 'ttbbto2l2nu': 11683236.0, 'ttbbto4q': 14012432.0
}

# --- PROCESS GROUPINGS ---
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ_genMatch', 'ttZ_nonMatch', 'ttH_genMatch', 'ttH_nonMatch'] 
mc_processes = bkg_processes + sig_processes
all_processes = mc_processes + ['data_obs'] if PLOT_DATA else mc_processes

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf/cc$', 'SingleTop': 'Single Top',
    'TTX': r'$t\bar{t}X$', 
    'ttZ_genMatch': r'$t\bar{t}Z$ (Matched)',   
    'ttZ_nonMatch': r'$t\bar{t}Z$ (Unmatched)', 
    'ttH_genMatch': r'$t\bar{t}H$ (Matched)',    
    'ttH_nonMatch': r'$t\bar{t}H$ (Unmatched)',  
    'data_obs': 'Data'
}

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {
    'ttZ_genMatch':'#832db6', 'ttZ_nonMatch':'#c671d4', 
    'ttH_genMatch':'#a96b59', 'ttH_nonMatch':'#d6a599'
} 
mc_colors = {**bkg_colors, **sig_colors}

# --- VARIABLES TO EXTRACT ---
cut_vars = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'MET_pt', 'muon_eta']
weight_vars = [
    'genWeight', 'topptWeight', 'ele_reco_sf', 'ele_id_sf', 
    'ele_trig_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf', 'puWeight'
]
encoder_vars = [f'encoderFeature{i+1}' for i in range(32)]
spanet_vars = ['sig_score', 'ttbb_score', 'ttlf_score'] 

# ==========================================
# 2. KINEMATIC CUTS & WEIGHT CALCULATOR
# ==========================================
def cuts(df_):
    return (
        (df_['n_ak4jets']  >= 5)       &
        (df_['n_b_outZH']  >= 2)       &
        (df_['ZH_pt']      >= 200)     &
        (((df_['ZH_M'] >= 50) & (df_['ZH_M'] <= 75)) | ((df_['ZH_M'] >= 145) & (df_['ZH_M'] <= 200))) &
        (df_['MET_pt']     > 20)       &
        (df_['ZH_bbvLscore'] > 0.56) 
    )

def getZhbbWeight(df_):
    """Calculates Final Weight. DF already contains pedantic dataset_norm_weight."""
    if 'dataset_norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 
        
    weight = df_['dataset_norm_weight'].copy()
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf', 'puWeight','ele_trig_sf', 'topptWeight']
    for sf in sfs:
        weight *= df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        
    return weight

# ==========================================
# 3. DATA LOADER (OPTIMIZED FOR NOMINAL)
# ==========================================
def get_mapped_proc(raw_proc):
    p = raw_proc.lower()
    if 'data' in p: return 'data_obs'
    if 'ttto' in p:
        if '__' not in p: return None  
        if 'tt+b' in p: return None    
        return 'TTBar'                 
        
    if 'ttbb' in p: 
        if '__' not in p: return None  
        if 'tt+b' in p: return 'tt_B'  
        return None                    
        
    if 'wjets' in p or 'dyjets' in p: return 'VJets'
    if 'qcd' in p: return 'QCD'
    
    if 'tth' in p:
        if '__non_genmatch' in p: return 'ttH_nonMatch'
        elif '__genmatch' in p: return 'ttH_genMatch'
        else: return 'ttH_genMatch'
    
    if 'ttz' in p or 'ttll' in p or 'ttnunu' in p:
        if '__non_genmatch' in p: return 'ttZ_nonMatch'
        elif '__genmatch' in p: return 'ttZ_genMatch'
        else: return 'ttZ_genMatch' 
            
    if 'singletop' in p: return 'SingleTop'
    if 'ttx' in p: return 'TTX'
    if 'vv' in p: return 'VV'
    return None

def load_and_cut_nominal_data(coffea_dir=COFFEA_DIR):
    tracked_data = {proc: [] for proc in all_processes}
    valid_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    print(f"--- Extracting from {len(valid_files)} nominal .coffea files ---")

    extract_list = cut_vars + weight_vars + encoder_vars + spanet_vars

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = get_mapped_proc(raw_proc)
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                
                if mapped_proc != 'data_obs':
                    xsec = TARGET_XSECS.get(clean_name, 0)
                    sum_gw = TARGET_GENWEIGHTS.get(clean_name)
                    if not sum_gw:
                        file_gw = genweight_dict.get(dataset, 1.0)
                        sum_gw = file_gw.get(dataset, 1.0) if isinstance(file_gw, dict) else file_gw
                    dataset_norm_weight = (xsec * LUMI * 1000) / sum_gw if sum_gw != 0 else 0
                else:
                    dataset_norm_weight = 1.0

                try:
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue 

                tmp_data = {}
                for var in extract_list:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    if dict_key in nom_dict:
                        tmp_data[var] = np.array(nom_dict[dict_key].value)

                if tmp_data: 
                    df = pd.DataFrame(tmp_data)
                    df['dataset_norm_weight'] = dataset_norm_weight
                    tracked_data[mapped_proc].append(df)

    data_dict = {}
    print("\n--- Applying Kinematic Cuts & K-Factors ---")
    for proc in all_processes:
        if tracked_data[proc]:
            df = pd.concat(tracked_data[proc], ignore_index=True)
            pre_cut = len(df)
            df = df[cuts(df)] 
            post_cut = len(df)
            
            # --- Calculate Custom SPANet Feature ---
            if all(v in df.columns for v in spanet_vars):
                denom = df['ttbb_score'] + df['ttlf_score'] + df['sig_score']
                df['final_score'] = np.where(denom > 0, df['sig_score'] / denom, 0)
            else:
                df['final_score'] = np.nan
            
            df['tot_weight'] = getZhbbWeight(df) if proc != 'data_obs' else 1.0
            if proc == 'tt_B': df['tot_weight'] *= TTBB_KFACTOR
            
            data_dict[proc] = df
            print(f" {proc:<10}: {pre_cut:>8} -> {post_cut:>8} events")
        else:
            data_dict[proc] = None
            print(f" {proc:<10}:        0 ->        0 events")
            
    return data_dict

# ==========================================
# 4. PLOTTING FUNCTION
# ==========================================
def plot_features_to_pdf(data_dict, variables_to_plot, output_filename="Features.pdf", n_bins=15):
    print(f"\n--- Generating Feature Plots: {output_filename} ---")
    
    with PdfPages(output_filename) as pdf:
        for chunk_start in range(0, len(variables_to_plot), 9):
            chunk_vars = variables_to_plot[chunk_start : chunk_start + 9]
            
            fig = plt.figure(figsize=(24, 28))
            outer_grid = fig.add_gridspec(3, 3, wspace=0.3, hspace=0.45)
            
            for idx, var_name in enumerate(chunk_vars):
                row, col = idx // 3, idx % 3
                
                inner_grid = outer_grid[row, col].subgridspec(3, 1, height_ratios=[4, 1, 1], hspace=0.1)
                ax = fig.add_subplot(inner_grid[0])
                rax1 = fig.add_subplot(inner_grid[1], sharex=ax)
                rax2 = fig.add_subplot(inner_grid[2], sharex=ax)
                
                ax.tick_params(labelbottom=False)
                rax1.tick_params(labelbottom=False)
                
                # --- Dynamic Binning ---
                all_vals = []
                for proc in all_processes:
                    df = data_dict.get(proc)
                    if df is not None and not df.empty and var_name in df.columns:
                        all_vals.append(df[var_name].dropna().values)
                
                if not all_vals:
                    ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
                    continue
                    
                global_vals = np.concatenate(all_vals)
                x_min, x_max = np.percentile(global_vals, [1, 99]) 
                bins = np.linspace(x_min, x_max, n_bins + 1)
                bin_centers = 0.5 * (bins[1:] + bins[:-1])

                mc_hists, mc_labels_list, mc_colors_list = [], [], []
                
                bkg_counts = np.zeros(n_bins)
                total_sig_counts = np.zeros(n_bins)   
                matched_sig_counts = np.zeros(n_bins) 
                bkg_err2 = np.zeros(n_bins) 
                
                # --- Process MC Histograms ---
                for proc in mc_processes:
                    df = data_dict.get(proc)
                    if df is None or df.empty or var_name not in df.columns:
                        continue
                        
                    mask = df[var_name].notna()
                    vals = np.clip(df[var_name][mask], bins[0], bins[-1])
                    weights = df['tot_weight'][mask]
                    
                    counts, _ = np.histogram(vals, bins=bins, weights=weights)
                    err2, _ = np.histogram(vals, bins=bins, weights=weights**2)
                    
                    mc_hists.append(counts)
                    mc_labels_list.append(f"{process_labels.get(proc, proc)}")
                    mc_colors_list.append(mc_colors[proc])
                    
                    if proc in sig_processes:
                        total_sig_counts += counts
                        #if 'genMatch' in proc:
                        matched_sig_counts += counts
                    elif proc in bkg_processes:
                        bkg_counts += counts
                        bkg_err2 += err2 

                total_mc_counts = bkg_counts + total_sig_counts

                # --- NEW: Print yields per bin for the final score ---
                if var_name == 'final_score':
                    print(f"\n--- Yields per bin for {var_name} ---")
                    for i in range(n_bins):
                        print(f"Bin [{bins[i]:.4f}, {bins[i+1]:.4f}]: Matched Signal = {matched_sig_counts[i]:.4f}, Background = {bkg_counts[i]:.4f}")

                # --- Plot Top Panel (Stacked MC) ---
                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                                 label=mc_labels_list, color=mc_colors_list, sort='yield')
                
                # --- Plot Middle Panel (S/B) ---
                with np.errstate(divide='ignore', invalid='ignore'):
                    s_over_b = matched_sig_counts / bkg_counts
                    s_over_b[np.isnan(s_over_b) | np.isinf(s_over_b)] = 0
                
                rax1.plot(bin_centers, s_over_b, 'bo', markersize=4) 
                rax1.axhline(0, color='black', linestyle='--', alpha=0.5)
                rax1.set_ylabel(r'$S/B$', fontsize=14)
                
                max_sb = np.max(s_over_b) if np.any(s_over_b > 0) else 1
                rax1.set_ylim(-0.05, max_sb * 1.3)
                rax1.grid(True, linestyle='--', alpha=0.5)

                # --- Plot Bottom Panel (DATA vs S/sqrt(B)) ---
                max_plot_val = np.max(total_mc_counts)
                
                if PLOT_DATA and 'data_obs' in data_dict and data_dict['data_obs'] is not None:
                    # Process & Plot Data
                    data_df = data_dict['data_obs']
                    mask = data_df[var_name].notna()
                    data_vals = np.clip(data_df[var_name][mask], bins[0], bins[-1])
                    
                    data_counts, _ = np.histogram(data_vals, bins=bins)
                    data_err = np.sqrt(data_counts)
                    data_yield = np.sum(data_counts)
                    
                    if data_yield > 0:
                        data_lbl = f"Data ({data_yield:.0f})"
                        hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                                     color='black', label=data_lbl, yerr=data_err)
                    
                    max_plot_val = max(max_plot_val, np.max(data_counts))
                    
                    # Ratio Panel (Data / MC)
                    with np.errstate(divide='ignore', invalid='ignore'):
                        ratio = data_counts / total_mc_counts
                        ratio_err = np.abs(data_err / total_mc_counts)
                        ratio[np.isnan(ratio) | np.isinf(ratio)] = 0
                    
                    rax2.errorbar(bin_centers, ratio, yerr=ratio_err, fmt='ko', markersize=4)
                    rax2.axhline(1, color='black', linestyle='--')
                    rax2.set_ylabel("Data / MC", fontsize=14)
                    rax2.set_ylim(0.5, 1.5) 

                else:
                    # Significance Panel
                    with np.errstate(divide='ignore', invalid='ignore'):
                        significance = matched_sig_counts / np.sqrt(bkg_counts) 
                        significance[np.isnan(significance) | np.isinf(significance)] = 0
                    
                    rax2.plot(bin_centers, significance, 'ko', markersize=4)
                    rax2.axhline(0, color='black', linestyle='--', alpha=0.5)
                    rax2.set_ylabel(r'$S/\sqrt{B}$', fontsize=14)
                    
                    max_sig = np.max(significance) if np.any(significance > 0) else 1
                    rax2.set_ylim(0, 3.5)

                # --- General Styling ---
                ax.set_ylabel("Events / Bin", fontsize=16)
                ax.set_title(f"Distribution of {var_name}", fontsize=18, pad=20)
                ax.legend(loc='upper right', ncol=2, fontsize=12) 
                
                ax.set_ylim(0, max_plot_val * 1.5 if max_plot_val > 0 else 100)
                
                rax2.set_xlabel('Feature Value', fontsize=16)
                rax2.grid(True, linestyle='--', alpha=0.5)
                
                hep.cms.label("Preliminary", data=PLOT_DATA, lumi=LUMI, ax=ax, com=13.6, fontsize=14)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig) 
            
    print(f"Finished! Plots saved successfully.")

# ==========================================
# 5. EXECUTION 
# ==========================================
if __name__ == "__main__":
    current_data_dict = load_and_cut_nominal_data(coffea_dir=COFFEA_DIR)
    
    plot_list = encoder_vars + ['final_score']
    
    output_pdf_name = "EncoderFeatures_Data_vs_MC.pdf" if PLOT_DATA else "EncoderFeatures_MC_Only.pdf"
    
    plot_features_to_pdf(current_data_dict, plot_list, output_filename=output_pdf_name)

--- Extracting from 6 nominal .coffea files ---

--- Applying Kinematic Cuts & K-Factors ---
 VJets     :    11558 ->       26 events
 QCD       :     2770 ->       21 events
 tt_B      :   479060 ->    15111 events
 TTBar     :  4443598 ->     8409 events
 SingleTop :   267346 ->     1221 events
 TTX       :  1976477 ->    66092 events
 ttZ_genMatch:    37227 ->      343 events
 ttZ_nonMatch:   203135 ->     1844 events
 ttH_genMatch:     6760 ->      390 events
 ttH_nonMatch:  1295572 ->    10417 events

--- Generating Feature Plots: EncoderFeatures_MC_Only.pdf ---

--- Yields per bin for final_score ---
Bin [0.0002, 0.0658]: Matched Signal = 15.5527, Background = 709.7810
Bin [0.0658, 0.1313]: Matched Signal = 3.9125, Background = 123.5585
Bin [0.1313, 0.1969]: Matched Signal = 2.3908, Background = 81.6870
Bin [0.1969, 0.2625]: Matched Signal = 2.6847, Background = 75.3573
Bin [0.2625, 0.3280]: Matched Signal = 2.6533, Background = 64.0171
Bin [0.3280, 0.3936]: Matched Signal = 2.27